In [7]:
!pip install nltk pandas numpy

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 1.9 MB/s eta 0:00:05
   ---- ----------------------------------- 1.0/9.9 MB 2.0 MB/s eta 0:00:05
   ----- ---------------------------------- 1.3/9.9 MB 2.1 MB/s eta 0:00:05
   ------- -------------------------------- 1.8/9.9 MB 2.0 MB/s eta 0:00:05
   -------- ------------------------------- 2.1/9.9 MB 2.0 MB/s eta 0:00:04
   ---------- ----------------------------- 2.6/9.9 MB 2.0 MB/s eta 0:00:04
   ----------- ---------------------------- 2.9/9.9 MB 2.0 MB/s eta 0:00:04
   ------------ --------------------------- 3.1/9.9 MB 1.9 MB/s eta 0:00:04
   -------------- ------------------------- 3.7/9.9 MB 1.9 MB/s eta 0:00:04
   --------------- ------------

In [15]:
!pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu126


In [2]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\navee\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# base_url="/content/drive/MyDrive/emoC/"

base_url="./Dataset_old/"

In [3]:
import numpy as np
import pandas as pd
import nltk
from sklearn.preprocessing import OneHotEncoder
def wordTokenize(text:str):
    return nltk.word_tokenize(text)

# Run everytime till this cell

In [ ]:
# Loading Dataset and Preprocessing
data = pd.read_csv(f'{base_url}emotion_sentimen_dataset.csv')
data.head()
df = data["text"].to_frame(name= "text")

def normalizeData(df):
    for index,row in df.iterrows():
        text = row['text']
        text = text.lower()
        text.strip()
        text.strip('''
        !"#$%&'()*+,-./:;<=>?@[]\\^_`{|}~
        ''')
        row['text'] = text
    return df

df = normalizeData(df)

vocabulary = {
    "<PAD>": 0,
    "<UNK>": 1}

currIdx=2

max_len = 60 # can increase to 80 for better context understanding

def buildVocabulary(df):
    global vocabulary
    global currIdx
    global max_len
    for idx,row in df.iterrows():
        text = row['text']
        text = wordTokenize(text)
        numSequence = []

        for word in text:
            if word not in vocabulary:
                vocabulary[word] = currIdx
                numSequence.append(currIdx)
                currIdx+=1
            else:
                numSequence.append(vocabulary[word])

        numSequence =  np.array(numSequence)
        if(len(numSequence)<max_len):
            padLen = max_len-len(numSequence)
            numSequence = np.append(numSequence ,  [vocabulary["<PAD>"]]*padLen)
        else:
            numSequence = numSequence[0:max_len]

        row['text'] = numSequence

# doing One hot encoding of labels
encoder = OneHotEncoder(sparse_output = False , handle_unknown = 'ignore')
emotion_column = data['Emotion'].tolist()
unique_emotion_list = np.unique(emotion_column)
emotion = np.array(emotion_column).reshape(-1,1)

y = encoder.fit_transform(emotion).astype(int)

y_df = pd.DataFrame(y , columns = unique_emotion_list  ) # final One Hot Encoded Labels
buildVocabulary(df)
print("Vocabulary Size : " , currIdx)
df.head(10)

Vocabulary Size :  75299


,text
0,"[2, 3, 4, 5, 6, 7, 8, 9, 10, 2, 11, 12, 7, 13,..."
1,"[15, 16, 17, 18, 19, 2, 11, 20, 0, 0, 0, 0, 0,..."
2,"[2, 21, 22, 7, 23, 2, 24, 7, 25, 26, 27, 28, 2..."
3,"[41, 42, 43, 44, 45, 46, 29, 2, 11, 47, 48, 49..."
4,"[2, 11, 56, 57, 58, 59, 60, 5, 61, 47, 35, 62,..."
5,"[2, 11, 67, 68, 2, 69, 31, 70, 18, 71, 35, 72,..."
6,"[77, 78, 79, 18, 80, 81, 82, 83, 84, 85, 31, 6..."
7,"[2, 88, 89, 90, 91, 44, 92, 29, 93, 17, 18, 94..."
8,"[2, 11, 78, 95, 96, 29, 97, 53, 78, 98, 99, 0,..."
9,"[2, 11, 47, 2, 100, 101, 102, 103, 104, 18, 27..."


In [ ]:
print(len(df))
print(df.head())
print(y_df.head())

839555
                                                text
0  [2, 3, 4, 5, 6, 7, 8, 9, 10, 2, 11, 12, 7, 13,...
1  [15, 16, 17, 18, 19, 2, 11, 20, 0, 0, 0, 0, 0,...
2  [2, 21, 22, 7, 23, 2, 24, 7, 25, 26, 27, 28, 2...
3  [41, 42, 43, 44, 45, 46, 29, 2, 11, 47, 48, 49...
4  [2, 11, 56, 57, 58, 59, 60, 5, 61, 47, 35, 62,...
   anger  boredom  empty  enthusiasm  fun  happiness  hate  love  neutral  \
0      0        0      0           0    0          0     1     0        0   
1      0        0      0           0    0          0     0     0        1   
2      0        0      0           0    0          0     0     0        1   
3      1        0      0           0    0          0     0     0        0   
4      0        0      0           0    0          0     0     0        1   

   relief  sadness  surprise  worry  
0       0        0         0      0  
1       0        0         0      0  
2       0        0         0      0  
3       0        0         0      0  
4       0        0    

In [ ]:
from sklearn.model_selection import train_test_split

# X = df.drop(columns = 'emotion_label'  ) # it will be treated as object because pandas will store this column as object even though this contains a sequence of int 64  (the column itself it object type ) and keras required integer type
# print(X)
# X = np.array(df['text'].tolist() , dtype = np.int32) # because keras required the input to be a 2d or 3d tensor and drop column
X = np.stack(df['text'].values).astype(np.int32)
X_train , X_test , y_train , y_test = train_test_split(X , y_df , test_size =0.2 , random_state = 42)

print("success")


success


In [ ]:
# Before Training -> Let's Save the X and y (preprocessed ), so as to avoid the  preprocessing everytime
import os
from joblib import dump
os.makedirs(f"{base_url}preProcessedInputs" , exist_ok = True)

dump({
    "X_train":X_train,
    "X_test":X_test,
    "y_train":y_train,
    "y_test":y_test,
    "vocab":vocabulary
},f"{base_url}preProcessedInputs/dataset.joblib")

['preProcessedInputs/dataset.joblib']

In [ ]:
# Training
# 1. Sequential Model
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding , GlobalAveragePooling1D , Dense

num_classes = 13 # unique number of classes or len(unique_emotion_list)
vocab_size = len(vocabulary) # actual Vocab Size
embedding_dim = 50
max_len = 10

# Defining Model
model = Sequential([
    Embedding(input_dim = vocab_size,
              output_dim = embedding_dim,
              input_length = max_len),
    GlobalAveragePooling1D(),
    Dense(64 , activation = 'relu'),
    Dense(num_classes , activation = 'softmax')
])

model.compile(
    optimizer = 'adam',
    loss= 'categorical_crossentropy',
    metrics = ['accuracy']
)
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_2      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:

history = model.fit(
    X_train,
    y_train,
    epochs =10,
    batch_size = 32,
    validation_data = (X_test , y_test)
)

loss , accuracy = model.evaluate(X_test , y_test)
print("Validation Accuracy : " , accuracy)


Epoch 1/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 724s 34ms/step - accuracy: 0.8924 - loss: 0.4585 - val_accuracy: 0.9930 - val_loss: 0.0374
Epoch 2/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 700s 33ms/step - accuracy: 0.9948 - loss: 0.0286 - val_accuracy: 0.9968 - val_loss: 0.0209
Epoch 3/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 747s 34ms/step - accuracy: 0.9974 - loss: 0.0136 - val_accuracy: 0.9977 - val_loss: 0.0113
Epoch 4/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 694s 33ms/step - accuracy: 0.9985 - loss: 0.0078 - val_accuracy: 0.9977 - val_loss: 0.0119
Epoch 5/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 740s 33ms/step - accuracy: 0.9988 - loss: 0.0060 - val_accuracy: 0.9987 - val_loss: 0.0075
Epoch 6/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 751s 33ms/step - accuracy: 0.9990 - loss: 0.0049 - val_accuracy: 0.9989 - val_loss: 0.0063
Epoch 7/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 736s 33ms/step - accuracy: 0.9991 - loss: 0.0041 - val_accuracy: 0.9990 - val_loss: 0.0062
Epoch 8/10
20989/20989 ━━━━━━━━━━━━━━━━━━━━ 753s 34ms/s

In [ ]:
# Saving the model and dumping the vocabulary, unique_emotion_list , max_len
import pickle
model.save(f'{base_url}sequential/emotion_model.keras')

with open(f'{base_url}vocab.pkl' , 'wb') as f:
  pickle.dump(vocabulary , f)

with open(f'{base_url}unique_emotion_list.pkl' , 'wb') as f:
  pickle.dump(unique_emotion_list , f)

with open(f'{base_url}max_len.pkl' , 'wb') as f:
  pickle.dump(max_len , f)

NameError: name 'model' is not defined

In [ ]:
# Predicting an Emotion Based on Our Trained Model (Sequential)
# 1. Doing the Same preprocessing
import pickle
from keras.models import load_model

vocabulary = {}
unique_emotion_list = []
max_len = 0
with open(f'{base_url}vocab.pkl' , 'rb') as f:
  vocabulary=pickle.load( f)

with open(f'{base_url}unique_emotion_list.pkl' , 'rb') as f:
  unique_emotion_list=pickle.load(f)

with open(f'{base_url}max_len.pkl' , 'rb') as f:
  max_len=pickle.load(f)

with open(f'{base_url}max_len.pkl' , 'rb') as f:
  max_len=pickle.load(f)
def preProcessInput(text):
  text = text.lower().strip().strip('''
        !"#$%&'()*+,-./:;<=>?@[]\\^_`{|}~
        ''')
  tokens = wordTokenize(text)
  sequence = []
  for word in tokens:
    if word in vocabulary:
      sequence.append(vocabulary[word])
    else:
      sequence.append(vocabulary["<UNK>"])
  if (len(sequence) < max_len):
    padlen = max_len - len(sequence)
    sequence.extend([vocabulary["<PAD>"]]*padlen)
  else:
    sequence = sequence[:max_len]
  return np.array(sequence).reshape(-1 , 1)




In [ ]:
text = input("Enter Some Text ")

processed = preProcessInput(text)
model = load_model(f'{base_url}sequential/emotion_model.keras')
prediction = model.predict(processed)

predicted_index = np.argmax(prediction, axis = 1)[0]
predicted_emotion = unique_emotion_list [predicted_index]
print("Predicted Emotion : " , predicted_emotion)

In [ ]:
# Optimization:
# 1. Make it More Denser
# 2. Use CNN, LSTM, RNN or Tranformers


In [6]:
# 2. Using CNN

from joblib import load
import torch
data = load(f'{base_url}preProcessedInputs/dataset.joblib')
X_train= data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
vocab = data["vocab"]

# We have saved y as one hot encoding dataframe -> convert it numpy array
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()
#  One-hot -> class indices
y_train = np.argmax(y_train , axis = 1)
y_test = np.argmax(y_test, axis = 1)
# Converting to Pytorch tensors

X_train= torch.tensor(X_train , dtype = torch.long)
X_test = torch.tensor(X_test , dtype = torch.long)
y_train = torch.tensor(y_train , dtype = torch.long)
y_test = torch.tensor(y_test , dtype = torch.long)



In [7]:
import torch.nn as nn
import torch.nn.functional as F

class TextCNN(nn.Module):
    def __init__(self , vocab_size, embed_dim, num_classes):
        super(TextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.conv1 = nn.Conv2d(1, 100, (3, embed_dim))
        self.conv2 = nn.Conv2d(1, 100, (4, embed_dim))
        self.conv3 = nn.Conv2d(1, 100, (5, embed_dim))
        self.fc = nn.Linear(300, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x=self.embedding(x)
        x= x.unsqueeze(1)
        x1 = F.relu(self.conv1(x)).squeeze(3)
        x1 = F.max_pool1d(x1, x1.size(2)).squeeze(2)

        x2 = F.relu(self.conv2(x)).squeeze(3)
        x2 = F.max_pool1d(x2 , x2.size(2)).squeeze(2)

        x3 = F.relu(self.conv3(x)).squeeze(3)
        x3 = F.max_pool1d(x3, x3.size(2)).squeeze(2)

        x = torch.cat((x1, x2, x3),1)
        x = self.dropout(x)

        return self.fc(x)

In [8]:
# initialize model
vocab_size = len(vocab) + 1
embed_dim = 100
num_classes = len(set(y_train.tolist()))
print(num_classes)

model = TextCNN(vocab_size , embed_dim , num_classes)

13


In [ ]:
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
# Using DataLoader (to avoid memory crash and pass data in batches)

train_dataset = TensorDataset(X_train , y_train)
train_loader = DataLoader(train_dataset , batch_size=64, shuffle=True)

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss= 0

    for batch_X, batch_y in train_loader:
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss +=loss.item()
    print(f" Epoch {epoch + 1 },  Loss: {total_loss:.4f}")


KeyboardInterrupt: 

In [4]:
# Saving
import torch

# torch.save(model.state_dict(), f"{base_url}text_cnn.pth")
torch.save({
    "model_state": model.state_dict(),
    "vocab_size": vocab_size,
    "embed_dim": embed_dim,
    "num_classes": num_classes
}, f"{base_url}cnn/text_cnn_full.pth")

NameError: name 'model' is not defined

In [ ]:
# label_map = list(unique_emotion_list)

# max_len = 60
# torch.save({
#     "model_state": model.state_dict(),
#     "vocab": vocab,
#     "label_map": label_map,
#     "config": {
#         "vocab_size": vocab_size,
#         "embed_dim": embed_dim,
#         "num_classes": num_classes,
#         "max_len": max_len
#     }
# }, f"{base_url}full_pipeline.pth")

In [9]:
# Predicting Emotion Based on CNN
# 1. Loading the model
import torch
checkpoint = torch.load(f"{base_url}cnn/text_cnn_full.pth" , map_location="cpu")
model = TextCNN(
    checkpoint["vocab_size"],
    checkpoint["embed_dim"],
    checkpoint["num_classes"]
)
# Load weights
model.load_state_dict(checkpoint["model_state"])
model.eval()

# 2. Doing the Same preprocessing
import pickle
from keras.models import load_model

vocabulary = {}
unique_emotion_list = []
max_len = 0
with open(f'{base_url}vocab.pkl' , 'rb') as f:
  vocabulary=pickle.load( f)

with open(f'{base_url}unique_emotion_list.pkl' , 'rb') as f:
  unique_emotion_list=pickle.load(f)

with open(f'{base_url}max_len.pkl' , 'rb') as f:
    max_len=pickle.load(f)
    


def preProcessInput(text):
  text = text.lower().strip().strip('''
        !"#$%&'()*+,-./:;<=>?@[]\\^_`{|}~
        ''')
  tokens = wordTokenize(text)
  sequence = []
  for word in tokens:
    if word in vocabulary:
      sequence.append(vocabulary[word])
    else:
      sequence.append(vocabulary["<UNK>"])
  if (len(sequence) < max_len):
    padlen = max_len - len(sequence)
    sequence.extend([vocabulary["<PAD>"]]*padlen)
  else:
    sequence = sequence[:max_len]
  return np.array(sequence)

# 3. Prediction
def predict(text):
  seq = preProcessInput(text)
  x = torch.tensor([seq], dtype = torch.long)

  with torch.no_grad():
    output = model(x)
    probs = F.softmax(output, dim=1)

  probs = probs.squeeze().tolist()
  result = {}
  for i, prob in enumerate(probs):
    emotion = unique_emotion_list[i]
    result[emotion]=round(prob*100 , 2)

  return result

text = input("Enter Some Text \n")
# Fails on text : i am extremely happy and a little sad
# i will fail in this exam due to zero support from the professor
# i will kill him

predict(text)
print(preProcessInput(text))

FileNotFoundError: [Errno 2] No such file or directory: './Dataset_old/cnn/text_cnn_full.pth'

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
model.eval()

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=256)

correct =0
total = 0

with torch.no_grad():
  for batch_X, batch_y in test_loader:
    outputs = model(batch_X)
    _, predicted = torch.max(outputs, 1)
    correct += (predicted == batch_y).sum().item()
    total +=batch_y.size(0)

accuracy = correct/total

print("Accuracy: ", accuracy)
# Accuracy:  0.9991602694284473 Last Cell Run

NameError: name 'model' is not defined

**Using BiLSTM**

In [4]:
# 3. Using BiLSTM
import torch
import torch.nn as nn

class TextLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super(TextLSTM, self).__init__()

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # BiLSTM layer
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Dropout
        self.dropout = nn.Dropout(0.5)

        # Fully connected layer
        self.fc = nn.Linear(hidden_dim * 2, num_classes)  # *2 because bidirectional

    def forward(self, x):
        # x: (batch_size, seq_len)

        x = self.embedding(x)
        # (batch_size, seq_len, embed_dim)

        lstm_out, _ = self.lstm(x)
        # (batch_size, seq_len, hidden_dim*2)

        # Take last time step
        out = lstm_out[:, -1, :]

        out = self.dropout(out)

        out = self.fc(out)

        return out


In [7]:
# Loading Vocabulary and unique emotion List
import pickle

vocabulary = {}
unique_emotion_list = []
max_len = 0
with open(f'{base_url}vocab.pkl' , 'rb') as f:
  vocabulary=pickle.load( f)

with open(f'{base_url}unique_emotion_list.pkl' , 'rb') as f:
  unique_emotion_list=pickle.load(f)

with open(f'{base_url}max_len.pkl' , 'rb') as f:
  max_len=pickle.load(f)

    
print("Max_len" , max_len)
max_len=60
vocab_size = len(vocabulary)
embed_dim = 100
hidden_dim = 128
num_classes = len(unique_emotion_list)

model_lstm = TextLSTM(vocab_size, embed_dim, hidden_dim, num_classes)

Max_len 10


In [12]:
# Actual Training Code for the LSTM MODEL
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# Device (important for Colab GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move model_lstm to device
model_lstm = model_lstm.to(device)

# Loss + Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)

# Dataset + DataLoader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Training loop
epochs = 20

for epoch in range(epochs):
    model_lstm.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        # Move to device
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        # Forward pass
        outputs = model_lstm(batch_X)
        loss = criterion(outputs, batch_y)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

KeyboardInterrupt: 

In [ ]:

# torch.save({
#     "lstm_state":model_lstm.state_dict()
# },f"{base_url}/LSTM/LSTM_MODEL.pth")

In [8]:
# Predicting Based on the BiLSTM
import torch.nn.functional as F
checkpoint = torch.load(f"{base_url}LSTM/LSTM_MODEL.pth")
model_lstm.load_state_dict(checkpoint["lstm_state"])
# setting the model to evaluation mode to avoid dropout
model_lstm.eval()

def preProcessInput(text):
  text = text.lower().strip().strip('''
        !"#$%&'()*+,-./:;<=>?@[]\\^_`{|}~
        ''')
  tokens = wordTokenize(text)
  sequence = []
  for word in tokens:
    if word in vocabulary:
      sequence.append(vocabulary[word])
    else:
      sequence.append(vocabulary["<UNK>"])
  if (len(sequence) < max_len):
    padlen = max_len - len(sequence)
    sequence.extend([vocabulary["<PAD>"]]*padlen)
  else:
    sequence = sequence[:max_len]
  return np.array(sequence)

def predict(text):
  seq = preProcessInput(text)
  x = torch.tensor([seq] , dtype= torch.long)

  with torch.no_grad():
    output = model_lstm(x)
    probs = F.softmax(output,dim=1)

    result = {}
    for i, prob in enumerate(probs.squeeze()):
      emotion = unique_emotion_list[i]
      result[emotion]=round(prob.item()*100 ,2)

    return result

text = input("Enter Some Text")
predict(text)


Enter Some Text im back to this niggling feeling of discontent and that overbearing sense of dread that starts to poke at me like a particularly annoying younger sibling


C:\Users\navee\AppData\Local\Temp\ipykernel_3204\3003718870.py:28: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  x = torch.tensor([seq] , dtype= torch.long)


{np.str_('anger'): 0.0,
 np.str_('boredom'): 0.0,
 np.str_('empty'): 0.0,
 np.str_('enthusiasm'): 0.0,
 np.str_('fun'): 0.0,
 np.str_('happiness'): 100.0,
 np.str_('hate'): 0.0,
 np.str_('love'): 0.0,
 np.str_('neutral'): 0.0,
 np.str_('relief'): 0.0,
 np.str_('sadness'): 0.0,
 np.str_('surprise'): 0.0,
 np.str_('worry'): 0.0}

In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score , f1_score
from torch.utils.data import DataLoader, TensorDataset

model_lstm.to(device)
model_lstm.eval()

test_dataset = TensorDataset(X_test , y_test)
test_loader = DataLoader(test_dataset, batch_size = 256)

all_preds = []
all_labels = []

with torch.no_grad():
  for batch_X, batch_y in test_loader:
    batch_X = batch_X.to(device)
    batch_Y = batch_y.to(device)

    outputs = model_lstm(batch_X)
    _, predicted = torch.max(outputs, 1)

    all_preds.extend(predicted.cpu().numpy())
    all_labels.extend(batch_y.cpu().numpy())

#Metrics
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average = "weighted")
recall = recall_score(all_labels, all_preds, average="weighted")
f1 = f1_score(all_labels, all_preds, average = "weighted")

print(f"Accuracy : {accuracy:.4f}" )
print(f"Precision : {precision:.4f}" )
print(f"Recall  : {recall : .4f}" )
print(f"F1 Score : {f1 :.4f}" )

Accuracy : 0.9987
Precision : 0.9988
Recall  :  0.9987
F1 Score : 0.9988


In [ ]:
# It Needs Training for Hindi words and Sentences

NameError: name 'rule' is not defined